In [9]:
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
_candidates = [
    _cwd / "dataset_construction" / "data" / "touche" / "touche_gemma4-v2-validated.csv",
    _cwd / "data" / "touche" / "touche_gemma4-v2-validated.csv",
]
INPUT_CSV = next((p for p in _candidates if p.exists()), _candidates[0])
DATA_ROOT = INPUT_CSV.parent.parent if INPUT_CSV.parent.name == "touche" else INPUT_CSV.parent
VB_CSV = DATA_ROOT / "dataset_negative_answer_mapped_clean.csv"
VB_CSV2 = DATA_ROOT / "dataset_positive_answer_mapped_clean.csv"
print(f"Touche input: {INPUT_CSV.resolve()}")
print(f"ValueBench clean: {VB_CSV.resolve()} (exists: {VB_CSV.exists()})")

df = pd.read_csv(INPUT_CSV)
print(f"Rows: {len(df)}, columns: {list(df.columns)}")
df["caa_suitable"].value_counts(dropna=False)

Touche input: /Users/hamidrezaei/Workspace/Steering_Geometry/dataset_construction/data/touche/touche_gemma4-v2-validated.csv
ValueBench clean: /Users/hamidrezaei/Workspace/Steering_Geometry/dataset_construction/data/dataset_negative_answer_mapped_clean.csv (exists: True)
Rows: 4000, columns: ['argument_id', 'split', 'stance', 'question', 'value', 'positive_answer', 'negative_answer', 'val_invokes_target_value', 'val_non_endorsing', 'val_rhetorically_coherent', 'val_quality', 'val_confidence', 'val_explanation', 'val_thinking', 'val_raw_response', 'caa_suitable']


caa_suitable
True     3741
False     259
Name: count, dtype: int64

In [10]:
s = df["caa_suitable"]
mask = (s == True) | (s.astype(str).str.strip().str.lower() == "true")
filtered = df.loc[
    mask, ["argument_id", "stance", "question", "value", "positive_answer", "negative_answer"]
].copy()
filtered.insert(0, "source", "touche")

print(f"After caa_suitable == True: {len(filtered)} rows")
filtered.head()

After caa_suitable == True: 3741 rows


,source,argument_id,stance,question,value,positive_answer,negative_answer
0,touche,A19454,in favor of,Should The vow of celibacy be abandoned?,Universalism: nature,the vow of celibacy puts undue pressure on pri...,The Church already maintains rigorous psycholo...
1,touche,A27090,against,Should we ban telemarketing?,Power: dominance,telemarketing is a cost-effective way of helpi...,Banning telemarketing would cause mass unemplo...
2,touche,A29354,in favor of,Intelligence tests bring more harm than good?,Universalism: tolerance,intelligence tests are a waste of money and di...,Standardized assessments provide essential dat...
3,touche,A27078,against,Should we subsidize embryonic stem cell research?,Universalism: nature,there are other avenues of stem cell research ...,"Current alternatives, such as induced pluripot..."
4,touche,A20137,in favor of,Should we stop the development of autonomous c...,Security: societal,the cars can be dangerous and cause accidents ...,Current data shows autonomous systems signific...


In [11]:
df_vb = pd.read_csv(VB_CSV)
n_vb = len(df_vb)
valuebench = pd.DataFrame({
    "source": ["valuebench"] * n_vb,
    "argument_id": ["NA"] * n_vb,
    "stance": ["NA"] * n_vb,
    "question": df_vb["question"],
    "value": df_vb["mapped_value"],
    "positive_answer": df_vb["positive_answer"],
    "negative_answer": df_vb["negative_answer"],
})

df_vb2 = pd.read_csv(VB_CSV2)
n_vb2 = len(df_vb2)
valuebench2 = pd.DataFrame({
    "source": ["valuebench"] * n_vb2,
    "argument_id": ["NA"] * n_vb2,
    "stance": ["NA"] * n_vb2,
    "question": df_vb2["question"],
    "value": df_vb2["mapped_value"],
    "positive_answer": df_vb2["positive_answer"],
    "negative_answer": df_vb2["negative_answer"],
})

# merged = pd.concat([filtered, valuebench, valuebench2], ignore_index=True)
merged = pd.concat([filtered, valuebench, valuebench2], ignore_index=True)
merged.insert(0, "id", range(len(merged)))
print(
    f"Touche rows: {len(filtered)}, "
    f"ValueBench negative rows: {len(valuebench)}, "
    f"ValueBench positive rows: {len(valuebench2)}, merged: {len(merged)}"
)
merged.head()

Touche rows: 3741, ValueBench negative rows: 594, ValueBench positive rows: 317, merged: 4652


,id,source,argument_id,stance,question,value,positive_answer,negative_answer
0,0,touche,A19454,in favor of,Should The vow of celibacy be abandoned?,Universalism: nature,the vow of celibacy puts undue pressure on pri...,The Church already maintains rigorous psycholo...
1,1,touche,A27090,against,Should we ban telemarketing?,Power: dominance,telemarketing is a cost-effective way of helpi...,Banning telemarketing would cause mass unemplo...
2,2,touche,A29354,in favor of,Intelligence tests bring more harm than good?,Universalism: tolerance,intelligence tests are a waste of money and di...,Standardized assessments provide essential dat...
3,3,touche,A27078,against,Should we subsidize embryonic stem cell research?,Universalism: nature,there are other avenues of stem cell research ...,"Current alternatives, such as induced pluripot..."
4,4,touche,A20137,in favor of,Should we stop the development of autonomous c...,Security: societal,the cars can be dangerous and cause accidents ...,Current data shows autonomous systems signific...


In [12]:
# Row counts by canonical `value`: total (merged) and split by source
print("=== Total rows per value (merged) ===")
print(merged["value"].value_counts().sort_index())
print(f"\nGrand total rows: {len(merged)}")

print("\n=== Rows per value × source (touche vs valuebench) ===")
print(pd.crosstab(merged["value"], merged["source"]))

print("\n=== Total rows per source ===")
print(merged["source"].value_counts())

=== Total rows per value (merged) ===
value
Achievement                   317
Benevolence: caring           296
Benevolence: dependability    217
Conformity: interpersonal     229
Conformity: rules             248
Face                          200
Hedonism                      206
Humility                      228
Power: dominance              231
Power: resources              207
Security: personal            262
Security: societal            198
Self-direction: action        234
Self-direction: thought       264
Stimulation                   218
Tradition                     214
Universalism: concern         252
Universalism: nature          224
Universalism: objectivity     191
Universalism: tolerance       216
Name: count, dtype: int64

Grand total rows: 4652

=== Rows per value × source (touche vs valuebench) ===
source                      touche  valuebench
value                                         
Achievement                    192         125
Benevolence: caring          

In [13]:
OUTPUT_CSV = DATA_ROOT / "final_dataset_v3.csv"
merged.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(merged)} rows → {OUTPUT_CSV.resolve()}")

Saved 4652 rows → /Users/hamidrezaei/Workspace/Steering_Geometry/dataset_construction/data/final_dataset_v3.csv


In [ ]:
# df_vb = pd.read_csv(VB_CSV)
# n_vb = len(df_vb)
# valuebench = pd.DataFrame({
#     "source": ["valuebench"] * n_vb,
#     "argument_id": ["NA"] * n_vb,
#     "stance": ["NA"] * n_vb,
#     "question": df_vb["question"],
#     "value": df_vb["mapped_value"],
#     "positive_answer": df_vb["positive_answer"],
#     "negative_answer": df_vb["negative_answer"],
# })
# merged = pd.concat([filtered, valuebench], ignore_index=True)
# merged.insert(0, "id", range(len(merged)))
# print(f"Touche rows: {len(filtered)}, ValueBench rows: {len(valuebench)}, merged: {len(merged)}")
# merged.head()